In [ ]:
import pandas as pd
df = pd.read_csv("diabetes.csv")
df

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,4,151.1,81.8,13.0,121.2,27.3,0.290,33,0
1,2,70.0,51.7,0.0,211.6,29.2,0.450,21,0
2,1,94.4,85.1,24.4,5.7,23.0,0.381,25,0
3,5,140.8,85.5,24.9,84.0,32.9,0.080,29,1
4,4,120.0,82.8,39.5,228.5,29.0,0.436,40,1
...,...,...,...,...,...,...,...,...,...
763,5,177.0,75.6,11.0,221.9,25.1,0.150,40,1
764,2,97.3,85.0,30.6,66.1,26.7,0.080,46,0
765,3,159.4,48.8,16.4,233.7,44.3,0.394,21,1
766,3,147.4,72.8,0.0,0.0,37.3,0.600,32,1


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    float64
 2   BloodPressure             768 non-null    float64
 3   SkinThickness             768 non-null    float64
 4   Insulin                   768 non-null    float64
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(6), int64(3)
memory usage: 54.1 KB


In [ ]:
import numpy as np
X = df.drop("Outcome", axis=1).values.astype(np.float32)
y = df["Outcome"].values.astype(np.float32)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
import torch
import torch.nn as nn

BATCH_SIZE = 32

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

train_dataset = torch.utils.data.TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = torch.utils.data.TensorDataset(X_test_tensor, y_test_tensor)



train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


In [ ]:
class DiabetesNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(8, 64),  nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 1),  nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x).squeeze(1)

model = DiabetesNet()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
criterion = nn.BCELoss()
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.5)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

num_epochs = 100

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels.squeeze(1))
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)

    scheduler.step()
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            predicted = (outputs > 0.5).float()

            total += labels.size(0)
            correct += (predicted == labels.squeeze(1)).sum().item()

    epoch_loss = running_loss / len(train_dataset)
    epoch_accuracy = 100 * correct / total

    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss}, Test Accuracy: {epoch_accuracy}%')

Epoch 1/100, Loss: 0.3545007623173903, Test Accuracy: 79.22077922077922%
Epoch 2/100, Loss: 0.3597129653738842, Test Accuracy: 79.22077922077922%
Epoch 3/100, Loss: 0.37034704863830964, Test Accuracy: 80.51948051948052%
Epoch 4/100, Loss: 0.3729724075390384, Test Accuracy: 79.87012987012987%
Epoch 5/100, Loss: 0.3847622997597685, Test Accuracy: 81.16883116883118%
Epoch 6/100, Loss: 0.37612858958275386, Test Accuracy: 81.16883116883118%
Epoch 7/100, Loss: 0.38489547949466335, Test Accuracy: 79.22077922077922%
Epoch 8/100, Loss: 0.3901126893413183, Test Accuracy: 79.22077922077922%
Epoch 9/100, Loss: 0.3573130441993378, Test Accuracy: 79.87012987012987%
Epoch 10/100, Loss: 0.36357847292959106, Test Accuracy: 80.51948051948052%
Epoch 11/100, Loss: 0.3727012140743119, Test Accuracy: 79.87012987012987%
Epoch 12/100, Loss: 0.36957209486914766, Test Accuracy: 79.87012987012987%
Epoch 13/100, Loss: 0.3791209628216995, Test Accuracy: 79.87012987012987%
Epoch 14/100, Loss: 0.3696757077394169, Te

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for xb, yb in train_loader:
        all_preds.extend((model(xb) >= 0.5).float().numpy())
        all_labels.extend(yb.numpy())

print(classification_report(all_labels, all_preds, target_names=["saglam", "Diabetli"]))

cm = confusion_matrix(all_labels, all_preds)
print(f"TN={cm[0,0]}  FP={cm[0,1]}  FN={cm[1,0]}  TP={cm[1,1]}")

              precision    recall  f1-score   support

      saglam       0.89      0.92      0.90       387
    Diabetli       0.86      0.80      0.83       227

    accuracy                           0.88       614
   macro avg       0.87      0.86      0.87       614
weighted avg       0.88      0.88      0.88       614

TN=357  FP=30  FN=46  TP=181
